# 02A — Block Sample and Label

Pipeline stage:
- load raw dataset
- minimal document filtering
- paragraph block segmentation
- block table materialization
- stratified block sampling
- LLM labeling
- labeled dataset export

Output directory

`output/02A_block_sample/`
- `docs.parquet`
- `block_sample.parquet`
- `block_labels.jsonl`
- `block_labels.parquet`
- `blocks/part_*.parquet`

In [1]:
!pip -q install pyarrow aiohttp tqdm

## Environment Setup

In [2]:
import os
import re
import json
import asyncio
from dataclasses import dataclass
from urllib.parse import urlparse

import numpy as np
import pandas as pd
import pyarrow.dataset as ds
from tqdm.auto import tqdm

import aiohttp

from google.colab import drive
drive.mount("/content/drive")

try:
    from google.colab import userdata
except Exception:
    userdata = None

BASE_DIR = "/content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen"
DATA_DIR = f"{BASE_DIR}/data"
OUT_DIR = f"{BASE_DIR}/output/02A_block_sample"

RAW_PARQUET = f"{DATA_DIR}/news_final_project.parquet"

DOCS_PARQUET = f"{OUT_DIR}/docs.parquet"
BLOCK_SAMPLE_PARQUET = f"{OUT_DIR}/block_sample.parquet"
BLOCK_LABELS_JSONL = f"{OUT_DIR}/block_labels.jsonl"
BLOCK_LABELS_PARQUET = f"{OUT_DIR}/block_labels.parquet"
BLOCKS_DIR = f"{OUT_DIR}/blocks"

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(BLOCKS_DIR, exist_ok=True)

Mounted at /content/drive


## Configuration

In [3]:
KEEP_LANGS = {"en"}

MIN_DOC_CHARS = 800
MAX_DOC_CHARS = 60000

MIN_BLOCK_CHARS = 30
MAX_BLOCKS_PER_DOC = 400
BLOCK_WRITE_BATCH = 200_000

TOP_DOMAIN_K = 40
BLOCK_SAMPLE_N = 12000

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

DEEPSEEK_BASE_URL = "https://api.deepseek.com"
DEEPSEEK_ENDPOINT = f"{DEEPSEEK_BASE_URL}/chat/completions"
DEEPSEEK_MODEL = "deepseek-chat"

LLM_CONCURRENCY = 64
LLM_TIMEOUT_S = 90
LLM_MAX_RETRIES = 6
JSONL_FLUSH_EVERY = 100
LABEL_MAX_CHARS = 1800

## Text Utilities

In [4]:
_ws_re = re.compile(r"\s+")
_ai_kw = re.compile(
    r"(?i)\b("
    r"ai|artificial intelligence|machine learning|deep learning|"
    r"generative ai|genai|large language model|llm|foundation model|"
    r"chatgpt|openai|anthropic|gemini|copilot|"
    r"computer vision|speech recognition|robotics"
    r")\b"
)

def normalize_spaces(text):
    return _ws_re.sub(" ", str(text or "")).strip()

def extract_domain(url):
    if not isinstance(url, str):
        return "NA"
    try:
        host = urlparse(url).netloc.lower()
        if host.startswith("www."):
            host = host[4:]
        return host if host else "NA"
    except Exception:
        return "NA"

def quarter_bucket(date):
    if not isinstance(date, str) or len(date) < 7:
        return "NA"
    try:
        y = date[:4]
        m = int(date[5:7])
        q = (m - 1) // 3 + 1
        return f"{y}Q{q}"
    except Exception:
        return "NA"

def ai_kw_flag(text):
    return int(bool(_ai_kw.search(str(text or ""))))

def segment_blocks(text):
    if not isinstance(text, str) or not text.strip():
        return []

    t = text.replace("\r\n", "\n").replace("\r", "\n").strip()

    blocks = [b.strip() for b in re.split(r"\n\s*\n+", t) if b.strip()]

    if len(blocks) <= 2:
        blocks = [b.strip() for b in t.split("\n") if b.strip()]

    blocks = [normalize_spaces(b) for b in blocks]
    blocks = [b for b in blocks if len(b) >= MIN_BLOCK_CHARS]

    if len(blocks) > MAX_BLOCKS_PER_DOC:
        blocks = blocks[:MAX_BLOCKS_PER_DOC]

    return blocks

## Load Dataset

In [5]:
df = pd.read_parquet(
    RAW_PARQUET,
    columns=["url", "date", "language", "title", "text"]
)

df["text"] = df["text"].fillna("").astype(str)
df["doc_len"] = df["text"].str.len()

mask = (df["doc_len"] >= MIN_DOC_CHARS) & (df["doc_len"] <= MAX_DOC_CHARS)

if KEEP_LANGS:
    mask &= df["language"].isin(KEEP_LANGS)

df = df.loc[mask].copy().reset_index(drop=True)
df["doc_id"] = np.arange(len(df), dtype=np.int64)

print("documents:", len(df))

documents: 197984


## Build Document Table

In [6]:
df["domain"] = df["url"].map(extract_domain)
df["time_bucket"] = df["date"].map(quarter_bucket)

docs = df[
    [
        "doc_id",
        "url",
        "date",
        "language",
        "title",
        "domain",
        "time_bucket",
        "doc_len",
    ]
].copy()

docs.to_parquet(DOCS_PARQUET, index=False)
print("wrote:", DOCS_PARQUET)

wrote: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/02A_block_sample/docs.parquet


## Block Segmentation

In [7]:
records = []
part = 0

for row in tqdm(df.itertuples(index=False), total=len(df), desc="Segmenting documents"):
    blocks = segment_blocks(row.text)

    for i, b in enumerate(blocks):
        records.append(
            {
                "doc_id": int(row.doc_id),
                "block_id": int(i),
                "block_uid": f"{int(row.doc_id)}:{int(i)}",
                "url": row.url,
                "date": row.date,
                "language": row.language,
                "title": row.title,
                "domain": row.domain,
                "time_bucket": row.time_bucket,
                "block_text": b,
                "block_len": len(b),
                "ai_kw_flag": ai_kw_flag(b),
            }
        )

    if len(records) >= BLOCK_WRITE_BATCH:
        path = f"{BLOCKS_DIR}/part_{part:04d}.parquet"
        pd.DataFrame(records).to_parquet(path, index=False)
        records = []
        part += 1

if records:
    path = f"{BLOCKS_DIR}/part_{part:04d}.parquet"
    pd.DataFrame(records).to_parquet(path, index=False)

print("block shards written")

Segmenting documents:   0%|          | 0/197984 [00:00<?, ?it/s]

block shards written


## Stratified Sampling

In [14]:
domain_counts = docs["domain"].value_counts()
top_domains = set(domain_counts.head(TOP_DOMAIN_K).index)

def domain_tier(x):
    return x if x in top_domains else "other"

docs["domain_tier"] = docs["domain"].map(domain_tier)

doc_strata = docs.groupby(["time_bucket", "domain_tier"]).size().reset_index(name="n")
doc_strata["w"] = doc_strata["n"] / doc_strata["n"].sum()

def alloc(df, n):
    keys = list(zip(df["time_bucket"], df["domain_tier"]))
    raw = (df["w"] * n).values
    base = np.floor(raw).astype(int)
    base = np.maximum(base, 1)

    while base.sum() > n:
        i = np.argmin(df["w"].values)
        if base[i] > 1:
            base[i] -= 1

    while base.sum() < n:
        i = np.argmax(raw - base)
        base[i] += 1

    return {keys[i]: base[i] for i in range(len(keys))}

targets = alloc(doc_strata, BLOCK_SAMPLE_N)

@dataclass
class Reservoir:
    k: int
    n: int
    items: list

def reservoir(res, item):
    res.n += 1
    if len(res.items) < res.k:
        res.items.append(item)
        return
    j = np.random.randint(0, res.n)
    if j < res.k:
        res.items[j] = item

res_map = {k: Reservoir(v, 0, []) for k, v in targets.items()}

dataset = ds.dataset(BLOCKS_DIR, format="parquet")
scanner = dataset.scanner()

for batch in tqdm(scanner.to_batches(), desc="Sampling blocks"):
    pdf = batch.to_pandas()
    pdf["domain_tier"] = pdf["domain"].map(domain_tier)

    for r in pdf.itertuples(index=False, name="Row"):
        key = (r.time_bucket, r.domain_tier)
        if key not in res_map:
            continue
        reservoir(res_map[key], r._asdict())

sample = []
for r in res_map.values():
    sample.extend(r.items)

sample_df = pd.DataFrame(sample).drop_duplicates("block_uid").reset_index(drop=True)

if len(sample_df) > BLOCK_SAMPLE_N:
    sample_df = sample_df.sample(BLOCK_SAMPLE_N, random_state=RANDOM_SEED).reset_index(drop=True)

sample_df.to_parquet(BLOCK_SAMPLE_PARQUET, index=False)

print("sample size:", len(sample_df))
print("wrote:", BLOCK_SAMPLE_PARQUET)

Sampling blocks: 0it [00:00, ?it/s]

sample size: 12000
wrote: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/02A_block_sample/block_sample.parquet


## DeepSeek Labeling

In [15]:
DEEPSEEK_API_KEY = os.environ.get("DEEPSEEK_API_KEY", "").strip()

if not DEEPSEEK_API_KEY and userdata is not None:
    try:
        DEEPSEEK_API_KEY = userdata.get("DEEPSEEK_API_KEY").strip()
    except Exception:
        pass

assert DEEPSEEK_API_KEY, "Missing DEEPSEEK_API_KEY"

SYSTEM_PROMPT = """
Return strict JSON.

Task:
Determine if the paragraph meaningfully discusses artificial intelligence.

Output:
{
  "is_ai_block": 1,
  "confidence": 0.92,
  "reason_short": "brief explanation"
}

Rules:
- Output JSON only.
- confidence must be in [0,1].
""".strip()

def user_prompt(row):
    text = str(row["block_text"])[:LABEL_MAX_CHARS]
    return (
        f"URL: {row['url']}\n"
        f"Domain: {row['domain']}\n"
        f"Date: {row['date']}\n"
        f"Title: {row['title']}\n\n"
        f"Block:\n{text}"
    )

def load_done_ids(path):
    if not os.path.exists(path):
        return set()
    done = set()
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                obj = json.loads(line)
                if "block_uid" in obj:
                    done.add(str(obj["block_uid"]))
            except Exception:
                continue
    return done

async def call_deepseek(session, payload, headers):
    for attempt in range(1, LLM_MAX_RETRIES + 1):
        try:
            async with session.post(
                DEEPSEEK_ENDPOINT,
                headers=headers,
                json=payload,
            ) as resp:
                txt = await resp.text()

                if resp.status in (429, 500, 502, 503, 504):
                    await asyncio.sleep(min(2 ** attempt, 20))
                    continue

                if resp.status != 200:
                    return {
                        "label_status": "error",
                        "error": f"http_{resp.status}",
                        "error_raw": txt[:500],
                    }

                data = json.loads(txt)
                content = data["choices"][0]["message"]["content"]

                try:
                    parsed = json.loads(content)
                    return {
                        "label_status": "ok",
                        "is_ai_block": int(parsed.get("is_ai_block", 0)),
                        "confidence": float(parsed.get("confidence", 0.0)),
                        "reason_short": str(parsed.get("reason_short", ""))[:200],
                    }
                except Exception:
                    return {
                        "label_status": "parse_error",
                        "error_raw": content[:500],
                    }

        except Exception as e:
            if attempt == LLM_MAX_RETRIES:
                return {
                    "label_status": "error",
                    "error": "exception",
                    "error_raw": str(e)[:500],
                }
            await asyncio.sleep(min(2 ** attempt, 20))

    return {
        "label_status": "error",
        "error": "max_retries",
    }

async def label_one(row, session, headers, sem):
    payload = {
        "model": DEEPSEEK_MODEL,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt(row)},
        ],
        "temperature": 0,
        "max_tokens": 180,
        "response_format": {"type": "json_object"},
    }

    async with sem:
        result = await call_deepseek(session, payload, headers)

    out = {
        "block_uid": str(row["block_uid"]),
        "doc_id": int(row["doc_id"]),
        "block_id": int(row["block_id"]),
        "is_ai_block": result.get("is_ai_block", None),
        "confidence": result.get("confidence", None),
        "reason_short": result.get("reason_short", ""),
        "label_status": result.get("label_status", "error"),
    }

    if "error" in result:
        out["error"] = result["error"]
    if "error_raw" in result:
        out["error_raw"] = result["error_raw"]

    return out

async def run_labeling(sample_df):
    done_ids = load_done_ids(BLOCK_LABELS_JSONL)
    todo = sample_df[~sample_df["block_uid"].astype(str).isin(done_ids)].copy().reset_index(drop=True)

    print("already labeled:", len(done_ids))
    print("remaining:", len(todo))

    if len(todo) == 0:
        return

    headers = {
        "Authorization": f"Bearer {DEEPSEEK_API_KEY}",
        "Content-Type": "application/json",
    }

    connector = aiohttp.TCPConnector(
        limit=LLM_CONCURRENCY,
        limit_per_host=LLM_CONCURRENCY,
        ttl_dns_cache=300,
    )
    timeout = aiohttp.ClientTimeout(total=LLM_TIMEOUT_S)
    sem = asyncio.Semaphore(LLM_CONCURRENCY)

    async with aiohttp.ClientSession(connector=connector, timeout=timeout) as session:
        tasks = [
            label_one(row, session, headers, sem)
            for row in todo.to_dict(orient="records")
        ]

        buffer = []
        with open(BLOCK_LABELS_JSONL, "a", encoding="utf-8") as f:
            for fut in tqdm(asyncio.as_completed(tasks), total=len(tasks), desc="Labeling blocks"):
                out = await fut
                buffer.append(out)

                if len(buffer) >= JSONL_FLUSH_EVERY:
                    for item in buffer:
                        f.write(json.dumps(item, ensure_ascii=False) + "\n")
                    f.flush()
                    buffer = []

            if buffer:
                for item in buffer:
                    f.write(json.dumps(item, ensure_ascii=False) + "\n")
                f.flush()

In [16]:
sample_df = pd.read_parquet(BLOCK_SAMPLE_PARQUET).copy()
await run_labeling(sample_df)

print("wrote:", BLOCK_LABELS_JSONL)

already labeled: 0
remaining: 12000


Labeling blocks:   0%|          | 0/12000 [00:00<?, ?it/s]

wrote: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/02A_block_sample/block_labels.jsonl


## Export Labeled Dataset

In [17]:
rows = []
with open(BLOCK_LABELS_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        try:
            rows.append(json.loads(line))
        except Exception:
            continue

labels = pd.DataFrame(rows)

for col, default_value in {
    "block_uid": None,
    "doc_id": pd.NA,
    "block_id": pd.NA,
    "is_ai_block": pd.NA,
    "confidence": pd.NA,
    "reason_short": "",
    "label_status": "error",
}.items():
    if col not in labels.columns:
        labels[col] = default_value

labels = labels.drop_duplicates(subset=["block_uid"], keep="last").copy()

labels["block_uid"] = labels["block_uid"].astype(str)
labels["doc_id"] = pd.to_numeric(labels["doc_id"], errors="coerce")
labels["block_id"] = pd.to_numeric(labels["block_id"], errors="coerce")
labels["is_ai_block"] = pd.to_numeric(labels["is_ai_block"], errors="coerce")
labels["confidence"] = pd.to_numeric(labels["confidence"], errors="coerce")
labels["reason_short"] = labels["reason_short"].fillna("").astype(str)
labels["label_status"] = labels["label_status"].fillna("error").astype(str)

sample_df = pd.read_parquet(BLOCK_SAMPLE_PARQUET).copy()
sample_df["block_uid"] = sample_df["block_uid"].astype(str)

required_sample_cols = [
    "doc_id",
    "block_id",
    "block_uid",
    "url",
    "date",
    "language",
    "title",
    "domain",
    "time_bucket",
    "block_text",
    "block_len",
    "ai_kw_flag",
]
missing_sample = [c for c in required_sample_cols if c not in sample_df.columns]
assert not missing_sample, f"Missing required columns in block_sample.parquet: {missing_sample}"

final = sample_df.merge(
    labels[
        [
            "block_uid",
            "is_ai_block",
            "confidence",
            "reason_short",
            "label_status",
        ]
    ],
    on="block_uid",
    how="left",
)

final["reason_short"] = final["reason_short"].fillna("").astype(str)
final["label_status"] = final["label_status"].fillna("error").astype(str)

final_cols = [
    "doc_id",
    "block_id",
    "block_uid",
    "url",
    "date",
    "language",
    "title",
    "domain",
    "time_bucket",
    "block_text",
    "block_len",
    "ai_kw_flag",
    "is_ai_block",
    "confidence",
    "reason_short",
    "label_status",
]

final = final[final_cols].copy()

final["doc_id"] = pd.to_numeric(final["doc_id"], errors="coerce").astype("Int64")
final["block_id"] = pd.to_numeric(final["block_id"], errors="coerce").astype("Int64")
final["block_len"] = pd.to_numeric(final["block_len"], errors="coerce").astype("Int64")
final["ai_kw_flag"] = pd.to_numeric(final["ai_kw_flag"], errors="coerce").astype("Int64")
final["is_ai_block"] = pd.to_numeric(final["is_ai_block"], errors="coerce").astype("Int64")
final["confidence"] = pd.to_numeric(final["confidence"], errors="coerce")

final.to_parquet(BLOCK_LABELS_PARQUET, index=False)

print("wrote:", BLOCK_LABELS_PARQUET)
print("shape:", final.shape)
print("\ncolumns:")
print(final.columns.tolist())
print("\nlabel_status counts:")
print(final["label_status"].value_counts(dropna=False))
print("\nis_ai_block counts:")
print(final["is_ai_block"].value_counts(dropna=False))

wrote: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/02A_block_sample/block_labels.parquet
shape: (12000, 16)

columns:
['doc_id', 'block_id', 'block_uid', 'url', 'date', 'language', 'title', 'domain', 'time_bucket', 'block_text', 'block_len', 'ai_kw_flag', 'is_ai_block', 'confidence', 'reason_short', 'label_status']

label_status counts:
label_status
ok    12000
Name: count, dtype: int64

is_ai_block counts:
is_ai_block
0    7778
1    4222
Name: count, dtype: Int64


## Schema Check

In [22]:
check_df = pd.read_parquet(BLOCK_LABELS_PARQUET)

required_cols = [
    "doc_id",
    "block_id",
    "block_uid",
    "block_text",
    "is_ai_block",
    "label_status",
]
missing = [c for c in required_cols if c not in check_df.columns]
assert not missing, f"Missing required columns: {missing}"

print("schema check passed")
display(check_df.head(10))

schema check passed


,doc_id,block_id,block_uid,url,date,language,title,domain,time_bucket,block_text,block_len,ai_kw_flag,is_ai_block,confidence,reason_short,label_status
0,151864,0,151864:0,https://www.benzinga.com/news/24/03/37416207/e...,2024-03-01,en,"Elon Musk Says 'Bring It On' To OpenAI, Micros...",benzinga.com,NA,"Elon Musk Says 'Bring It On' To OpenAI, Micros...",5628,1,1,0.85,"Mentions OpenAI and humanoid robots, implying ...",ok
1,78975,2,78975:2,https://www.benzinga.com/markets/equities/24/0...,2024-08-09,en,"PLTR, SOUN, U, TTD, TSLA: Top 5 Trending Stock...",benzinga.com,NA,This story was generated using Benzinga Neuro ...,1159,0,1,0.85,Block includes AI-related tags and mentions AI...,ok
2,145689,6,145689:6,https://www.benzinga.com/analyst-ratings/analy...,2024-08-30,en,Marvell Technology Posts First Beat-And-Raise ...,benzinga.com,NA,"""Supported by strong bookings and a robust bac...",166,0,1,0.85,Mentions 'AI Story' in context of company perf...,ok
3,26043,1,26043:1,https://www.benzinga.com/markets/tech/25/07/46...,2025-07-19,en,Forget Chatbots—Perplexity's Aravind Srinivas ...,benzinga.com,NA,"Dan Ives Urges Apple ‘To Make A Move,' Saying ...",127,0,1,0.85,"The block discusses Perplexity, an AI company,...",ok
4,119343,2,119343:2,https://www.benzinga.com/markets/tech/25/06/45...,2025-06-12,en,Jamie Dimon Recalls First Palantir Meeting: 'H...,benzinga.com,NA,Revolutionary Manufacturing Approach – Inspire...,190,0,0,0.95,"Block discusses manufacturing and housing, not...",ok
5,145504,23,145504:23,https://www.benzinga.com/news/22/01/25010290/a...,2022-01-12,en,"Apple Inc. (NASDAQ:AAPL), HON HAI PREC INDS RE...",benzinga.com,NA,Fintech Focus A daily collection of all things...,161,0,0,0.85,Block describes general fintech and SPAC topic...,ok
6,64292,1,64292:1,https://www.benzinga.com/markets/tech/25/12/49...,2025-12-21,en,Gene Munster Argues OpenAI Is 'Undervalued' Ev...,benzinga.com,NA,"In a related blog post, Munster cited a Wall S...",3053,1,1,0.95,"Discusses AI valuation, growth, and industry i...",ok
7,171657,1,171657:1,https://www.benzinga.com/pressreleases/22/02/g...,2022-08-21,en,"Responding to Community Demand, Comet Announce...",benzinga.com,NA,"Responding to Community Demand, Comet Announce...",2646,1,1,0.95,The paragraph explicitly discusses machine lea...,ok
8,138741,0,138741:0,https://www.benzinga.com/markets/tech/25/09/47...,2025-09-15,en,"Everything You Need To Know About iOS 26, Appl...",benzinga.com,NA,"Everything You Need To Know About iOS 26, Appl...",4255,1,0,0.95,Block consists of website navigation elements ...,ok
9,64293,0,64293:0,https://www.benzinga.com/news/24/09/40994173/r...,2024-09-24,en,"Russia, Iran Overcome Language Barriers To Spr...",benzinga.com,NA,"Russia, Iran Overcome Language Barriers To Spr...",5054,1,1,0.85,Block mentions AI improving foreign election i...,ok
